# 11 Decision heads compared: why logistic regression?

Every notebook so far puts a logistic regression on frozen encoder embeddings. That was a choice, not a finding. This notebook tests it: the same embeddings, six different heads, the same 200 test messages.

A **head** is the small model on top of the encoder that turns an embedding into a probability for each route. The six: logistic regression (our default), a linear support vector machine, k-nearest neighbors, a small neural network (one hidden layer), gradient-boosted trees, and a **prototype** classifier (the mean embedding of each route, scored by cosine similarity).

For each head we report accuracy with a Wilson interval, the paired comparison with logistic regression, the time the head itself takes for one message, and calibration error before and after temperature scaling (notebook 03). Nothing here trains the encoder, so the notebook runs in under a minute.

What it can and cannot show: 200 test messages with 6 to 15 errors per head cannot separate heads that differ by a point or two. A difference is reported only if the interval and the paired test both show it.

In [1]:
import json, time
import numpy as np
from pathlib import Path
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
ENC = r01["_best"]
tr, ca, te = m.load("train"), m.load("calibration"), m.load("test")
_, ytr = m.texts_labels(tr); _, yca = m.texts_labels(ca); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
Xtr, Xca, Xte = E["train"], E["calibration"], E["test"]
norm = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True)
print(f"encoder {ENC}: {Xtr.shape[1]}-dimensional embeddings, {len(ytr)} training, {len(yca)} calibration, {len(yte)} test messages")

encoder bge-small: 384-dimensional embeddings, 800 training, 200 calibration, 200 test messages


## Fit the six heads

The **prototype** head has no training beyond averaging: each route's prototype is the mean of its normalized training embeddings, and a message is scored by cosine similarity to each prototype (scaled by 20 so the scores spread out). Every other head is trained on the 800 training messages with default or lightly set options and no tuning, so no head gets an advantage from a search that the others did not get.

In [2]:
heads = {
    "logistic regression": LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(Xtr, ytr),
    "linear SVM": LinearSVC(C=1.0, max_iter=20000).fit(Xtr, ytr),
    "5 nearest neighbors": KNeighborsClassifier(5, metric="cosine").fit(Xtr, ytr),
    "neural net (128 hidden)": MLPClassifier((128,), max_iter=500, random_state=m.SEED).fit(Xtr, ytr),
    "gradient-boosted trees": HistGradientBoostingClassifier(max_iter=100, random_state=m.SEED).fit(Xtr, ytr),
}
protos = norm(np.stack([norm(Xtr[ytr == k]).mean(0) for k in range(len(m.ROUTES))]))

def scores(name, X):
    """One row of route scores per message (logits). Probabilities that are exactly 0 get a small floor before the log."""
    if name == "prototype (class means)": return norm(X) @ protos.T * 20
    h = heads[name]
    if name in ("logistic regression", "linear SVM"): return h.decision_function(X)
    return np.log(h.predict_proba(X) + 1e-3)

NAMES = list(heads) + ["prototype (class means)"]

**What the next cell does.** For each head it predicts the test messages and prints accuracy with its Wilson interval. It then counts the messages where logistic regression is right and the head wrong (`n10`) and the reverse (`n01`), with an exact McNemar p-value (notebook 04), and times the head alone on one message at a time (median of 200 calls).

In [3]:
ok = {n: scores(n, Xte).argmax(1) == yte for n in NAMES}
res = {}
for n in NAMES:
    k = int(ok[n].sum()); lo, hi = m.wilson(k, len(yte))
    n10 = int((ok["logistic regression"] & ~ok[n]).sum()); n01 = int((~ok["logistic regression"] & ok[n]).sum())
    p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    ts = []
    for i in range(len(yte)):
        t0 = time.perf_counter(); scores(n, Xte[i:i + 1]); ts.append(time.perf_counter() - t0)
    res[n] = dict(acc=k / len(yte), wilson=[lo, hi], lr_only=n10, head_only=n01, p=float(p), head_ms=float(np.median(ts) * 1000))
    print(f"{n:25s} acc {k / len(yte):.3f} [{lo:.3f}, {hi:.3f}]  LR-only right {n10:2d}, head-only right {n01:2d}, p = {p:.3f}  head time {np.median(ts) * 1000:.2f} ms")
print(f"\nfor scale: one forward pass of the encoder takes about {r01['_latency_batch1'][ENC]['median_ms']:.0f} ms per message (notebook 01)")

logistic regression       acc 0.970 [0.936, 0.986]  LR-only right  0, head-only right  0, p = 1.000  head time 0.11 ms
linear SVM                acc 0.970 [0.936, 0.986]  LR-only right  1, head-only right  1, p = 1.000  head time 0.11 ms


5 nearest neighbors       acc 0.880 [0.828, 0.918]  LR-only right 20, head-only right  2, p = 0.000  head time 2.35 ms
neural net (128 hidden)   acc 0.975 [0.943, 0.989]  LR-only right  1, head-only right  2, p = 1.000  head time 0.19 ms


gradient-boosted trees    acc 0.940 [0.898, 0.965]  LR-only right  7, head-only right  1, p = 0.070  head time 12.79 ms
prototype (class means)   acc 0.950 [0.910, 0.973]  LR-only right  4, head-only right  0, p = 0.125  head time 0.01 ms

for scale: one forward pass of the encoder takes about 31 ms per message (notebook 01)


**Goal.** Find out whether a different head would have been a better choice than the default, and what each choice costs. The answer that matters for a router is whether any head is clearly more accurate, because the heads all cost less than a millisecond next to an encoder pass of tens of milliseconds.

**Reading the output.** Four heads are within noise of logistic regression (97.0%): the linear SVM (97.0%), the neural net (97.5%, right on 2 messages where logistic regression is wrong and wrong on 1 where it is right) and the prototype head (95.0%, 4 messages worse, p = 0.125). Gradient-boosted trees are 94.0% (7 messages worse, p = 0.070), which does not clear the usual 0.05 bar at this sample size but points the same way as the two small studies that found trees no better on small text sets. Only 5 nearest neighbors is clearly worse: 88.0%, wrong on 20 messages where logistic regression is right and right on 2 where it is wrong. The head itself costs about 0.02 to 0.2 ms for every head except nearest neighbors (about 2 ms) and trees (about 13 ms, under half an encoder pass). These times include the library call overhead, which dominates for models this small, and vary by a fraction from run to run on this shared laptop.

## Calibration: can each head's confidence be fixed with one number?

The conformal gate and the release thresholds in notebooks 03 and 05 need probabilities that mean what they say. Some heads (logistic regression, SVM, prototypes) produce scores on an arbitrary scale and need temperature scaling; others (neural net, trees, nearest neighbors) produce probabilities that are closer to a scale of their own. The next cell measures expected calibration error (ECE, 10 bins) before and after dividing the scores by one temperature fitted on the 200 calibration messages.

In [4]:
def ece(P, y, bins=10):
    conf, pred = P.max(1), P.argmax(1); acc = pred == y
    edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any(): tot += s.mean() * abs(acc[s].mean() - conf[s].mean())
    return tot
def nll(T, Z, y): return -(Z / T - logsumexp(Z / T, axis=1, keepdims=True))[np.arange(len(y)), y].mean()

for n in NAMES:
    Zc, Zt = scores(n, Xca), scores(n, Xte)
    T = minimize_scalar(lambda t: nll(t, Zc, yca), bounds=(0.02, 50), method="bounded").x
    res[n].update(T=float(T), ece_raw=float(ece(softmax(Zt, 1), yte)), ece_scaled=float(ece(softmax(Zt / T, 1), yte)))
    print(f"{n:25s} T = {T:5.2f}   ECE before {res[n]['ece_raw']:.3f} -> after {res[n]['ece_scaled']:.3f}")
(RESULTS / "11_decision_heads.json").write_text(json.dumps(dict(encoder=ENC, heads=res), indent=2, default=float))

logistic regression       T =  0.25   ECE before 0.440 -> after 0.029
linear SVM                T =  0.19   ECE before 0.537 -> after 0.017
5 nearest neighbors       T =  1.55   ECE before 0.056 -> after 0.135
neural net (128 hidden)   T =  0.92   ECE before 0.039 -> after 0.031
gradient-boosted trees    T =  1.30   ECE before 0.024 -> after 0.055
prototype (class means)   T =  0.17   ECE before 0.574 -> after 0.038


2125

**What we got.** The three heads that produce scores on an arbitrary scale (logistic regression, SVM, prototypes) start with ECE of 0.44, 0.54 and 0.57 and end at 0.03, 0.02 and 0.04 after dividing by a temperature below 1 (0.25, 0.19, 0.17): they are underconfident and one number fixes it. The neural net is nearly calibrated to begin with (0.039) and ends at 0.031. Nearest neighbors and trees get worse (0.056 to 0.135 and 0.024 to 0.055): their probabilities are already on a usable scale (counts of neighbors, boosted scores), and a single temperature cannot repair coarse values. Temperature scaling is a fix for a particular failure of linear heads, not for every head. Logistic regression ends at 0.029 here against 0.044 in notebook 03, because this cell fits the temperature on all 200 calibration messages and notebook 03 on a 100-message half, keeping the other half for the conformal cutoff.

**What this says about our choice.** Logistic regression was not the most accurate head here (the neural net was 0.5 points higher, inside the noise), but it was among the best, it is the cheapest to calibrate, and it keeps the conformal gate simple. This is one encoder, one split and untuned defaults, so it supports "logistic regression is a sound default" and does not support "no other head can win". A tree or neural head with a few thousand rows, or with structured features beside the text, is a case to measure rather than rule out.

## Why not a Netflix-style scoring head?

The scoring head in Netflix's GenRec (Li et al., arXiv:2608.10257, https://arxiv.org/abs/2608.10257) scores each candidate item by combining the model's hidden state with a learned embedding of that item, then applies a softmax over the catalogue. With eight fixed routes, a learned embedding per route and a dot product with the message embedding is the same computation as a linear head: logistic regression already is that model. The design pays off when the candidate set is large or changes, such as products, documents or tools, because a new candidate needs only an embedding, not a retrained output layer. Notebook 02's zero-shot router uses this idea with fixed route-description embeddings and no learning. A learned version for a changing candidate set is not run in this workshop.